## Preprocessing


In [ ]:
!pip install opencv-python numpy pillow pandas tqdm matplotlib

In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
original_folder = "images"
preprocessed_folder = "preprocessed_images"
os.makedirs(preprocessed_folder, exist_ok=True)

def read_image_anyformat(path):
    try:
        img = Image.open(path).convert("RGB")
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    except:
        return None

def preprocess_image(img):
    img = cv2.fastNlMeansDenoisingColored(img,None,5,5,7,21)
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    clahe = cv2.createCLAHE(2.0,(8,8))
    ycrcb[:,:,0] = clahe.apply(ycrcb[:,:,0])
    img = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
    kernel = np.array([[0,-0.25,0],[-0.25,2,-0.25],[0,-0.25,0]],np.float32)
    img = cv2.filter2D(img,-1,kernel)
    return np.clip(img,0,255).astype(np.uint8)

first_image_displayed = False  

for filename in os.listdir(original_folder):
    img_orig = read_image_anyformat(os.path.join(original_folder, filename))
    if img_orig is None:
        continue

    img_pre = preprocess_image(img_orig)
    save_path = os.path.join(preprocessed_folder, filename)
    cv2.imwrite(save_path, img_pre)

    if not first_image_displayed:
        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1)
        plt.imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        plt.title("Original")
        plt.axis('off')

        plt.subplot(1,2,2)
        plt.imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
        plt.title("Preprocessed")
        plt.axis('off')
        plt.show()
        first_image_displayed = True

print("✅ Preprocessing done and images saved to preprocessed_images")


## DNN Face Detector

In [ ]:
modelFile = "res10_300x300_ssd_iter_140000.caffemodel"
configFile = "deploy.prototxt.txt"
net = cv2.dnn.readNetFromCaffe(configFile, modelFile)

def detect_faces(img, conf=0.2):
    h,w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(img,(300,300)),1,(300,300),(104,177,123))
    net.setInput(blob)
    det = net.forward()
    faces=[]
    for i in range(det.shape[2]):
        if det[0,0,i,2]>conf:
            box = det[0,0,i,3:7]*np.array([w,h,w,h])
            x1,y1,x2,y2 = box.astype(int)
            x1,y1 = max(0,x1),max(0,y1)
            x2,y2 = min(w,x2),min(h,y2)
            faces.append((x1,y1,x2,y2))
    return faces

img_pre = preprocess_image(read_image_anyformat(os.path.join(original_folder, os.listdir(original_folder)[0])))
faces = detect_faces(img_pre)
detect_img = img_pre.copy()
for x1,y1,x2,y2 in faces:
    cv2.rectangle(detect_img,(x1,y1),(x2,y2),(0,255,0),2)

filename = os.listdir(original_folder)[2]
img_orig = read_image_anyformat(os.path.join(original_folder, filename))
img_pre = preprocess_image(img_orig)

faces_orig = detect_faces(img_orig)
faces_pre = detect_faces(img_pre)

detect_orig = img_orig.copy()
detect_pre = img_pre.copy()

for x1,y1,x2,y2 in faces_orig:
    cv2.rectangle(detect_orig,(x1,y1),(x2,y2),(0,255,0),2)

for x1,y1,x2,y2 in faces_pre:
    cv2.rectangle(detect_pre,(x1,y1),(x2,y2),(0,255,0),2)

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(detect_orig, cv2.COLOR_BGR2RGB))
plt.title("Detected on Original")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(cv2.cvtColor(detect_pre, cv2.COLOR_BGR2RGB))
plt.title("Detected on Preprocessed")
plt.axis('off')

plt.show()
print(f"✅ Detected {len(faces_orig)} faces on original, {len(faces_pre)} faces on preprocessed")



## kernels

In [ ]:
kernels = list(range(3, 33, 2)) 

## Gaussian Blur 

In [ ]:
output_root = "output_kernels/gaussian"

for k in kernels:
    os.makedirs(os.path.join(output_root, f'k{k}'), exist_ok=True)

for filename in tqdm(os.listdir(original_folder)):
    img_orig = read_image_anyformat(os.path.join(original_folder, filename))
    if img_orig is None:
        continue
    img_pre = preprocess_image(img_orig)
    faces_pre = detect_faces(img_pre)
    
    for k in kernels:
       temp = img_pre.copy()
       for x1, y1, x2, y2 in faces_pre:
            face = temp[y1:y2, x1:x2]
            face = cv2.GaussianBlur(face, (k, k), 0)
            temp[y1:y2, x1:x2] = face
       cv2.imwrite(os.path.join(output_root, f'k{k}', filename), temp)
print("✅ Done! gaussian Blur applied and saved successfully.")



## Average Blur 

In [ ]:
output_root = "output_kernels/average"

for k in kernels:
    os.makedirs(os.path.join(output_root, f'k{k}'), exist_ok=True)

for filename in tqdm(os.listdir(original_folder)):
    img_orig = read_image_anyformat(os.path.join(original_folder, filename))
    if img_orig is None:
        continue
    img_pre = preprocess_image(img_orig)
    faces_pre = detect_faces(img_pre)
    
    for k in kernels:
       temp = img_pre.copy()
       for x1, y1, x2, y2 in faces_pre:
            face = temp[y1:y2, x1:x2]
            face = cv2.blur(face, (k, k))
            temp[y1:y2, x1:x2] = face
       cv2.imwrite(os.path.join(output_root, f'k{k}', filename), temp)
print("✅ Done! average Blur applied and saved successfully.")



## Median Blur 

In [ ]:
output_root = "output_kernels/median"

for k in kernels:
    os.makedirs(os.path.join(output_root, f'k{k}'), exist_ok=True)

for filename in tqdm(os.listdir(original_folder)):
    img_orig = read_image_anyformat(os.path.join(original_folder, filename))
    if img_orig is None:
        continue

    img_pre = preprocess_image(img_orig)
    faces_pre = detect_faces(img_pre)

    for k in kernels:
        temp = img_pre.copy()
        for x1,y1,x2,y2 in faces_pre:
            face = temp[y1:y2,x1:x2]
            face = cv2.medianBlur(face, k)
            temp[y1:y2,x1:x2] = face
        cv2.imwrite(os.path.join(output_root, f'k{k}', filename), temp)

print("✅ Done! Median Blur applied and saved successfully.")


## Blur Score

In [ ]:
preprocessed_folder = "preprocessed_images"

kernels_root = "output_kernels"  

filters = ["gaussian", "average", "median"]
kernels = list(range(3, 32, 2))  

def blur_intensity(original, blurred):
    return np.mean((original.astype(np.float32) - blurred.astype(np.float32))**2)

results = pd.DataFrame(index=kernels, columns=filters)

for k in kernels:
    for f in filters:
        blurred_folder = os.path.join(kernels_root, f, f'k{k}')
        intensities = []
        for filename in tqdm(os.listdir(preprocessed_folder), desc=f"{f} k{k}"):
            pre_img_path = os.path.join(preprocessed_folder, filename)
            blurred_img_path = os.path.join(blurred_folder, filename)

            if not os.path.exists(pre_img_path) or not os.path.exists(blurred_img_path):
                continue

            pre_img = cv2.imread(pre_img_path)
            blurred_img = cv2.imread(blurred_img_path)

            intensity = blur_intensity(pre_img, blurred_img)
            intensities.append(intensity)

        results.at[k, f] = np.mean(intensities) if intensities else None

print("✅ Blur intensity table:")
print(results)


## Blur Levels

In [ ]:
min_val = results.min().min()
max_val = results.max().max()

bins = np.linspace(min_val, max_val, 6) 

levels = results.applymap(lambda x: min(int(np.digitize(x, bins)), 5))

print("✅ Blur levels (1=least, 5=most):")
print(levels)


In [ ]:
blur_mapping = {}
for blur in levels.columns:
    blur_mapping[blur] = {}
    for lvl in range(1,6):
        kernels_for_lvl = [k for k in levels.index if levels.loc[k, blur] == lvl]
        blur_mapping[blur][lvl] = kernels_for_lvl

import pprint
pprint.pprint(blur_mapping)


## Main

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

original_folder = r"images"
preprocessed_folder = r"preprocessed_images"
output_folder = r"output"

os.makedirs(preprocessed_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

user_level = input("Enter blur level you want (0 for all levels, 1-5 for specific): ")
try:
    user_level = int(user_level)
    if user_level not in range(0,6):
        raise ValueError
except:
    print("Invalid input. Defaulting to level 1.")
    user_level = 1

for filename in os.listdir(original_folder):
    orig_path = os.path.join(original_folder, filename)
    img_orig = cv2.imread(orig_path)
    if img_orig is None:
        continue

    img_pre = preprocess_image(img_orig)
    pre_path = os.path.join(preprocessed_folder, filename)
    cv2.imwrite(pre_path, img_pre)

    faces_ori_pre = detect_faces(img_orig)
    faces_pre = detect_faces(img_pre)

    img_det_orig = img_orig.copy()
    for x1, y1, x2, y2 in faces_ori_pre:
        cv2.rectangle(img_det_orig, (x1, y1), (x2, y2), (0,255,0), 2)

    img_det_pre = img_pre.copy()
    for x1, y1, x2, y2 in faces_pre:
        cv2.rectangle(img_det_pre, (x1, y1), (x2, y2), (0,255,0), 2)

    if user_level == 0:
        total_cols = 3
        total_rows = 2 + 5
        plt.figure(figsize=(total_cols*4, total_rows*4))
        img_count = 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        plt.title('Original')
        plt.axis('off')
        img_count += 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
        plt.title('Preprocessed')
        plt.axis('off')
        img_count += 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.axis('off')
        img_count += 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.imshow(cv2.cvtColor(img_det_orig, cv2.COLOR_BGR2RGB))
        plt.title('Detected Faces (Original)')
        plt.axis('off')
        img_count += 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.imshow(cv2.cvtColor(img_det_pre, cv2.COLOR_BGR2RGB))
        plt.title('Detected Faces (Preprocessed)')
        plt.axis('off')
        img_count += 1

        plt.subplot(total_rows, total_cols, img_count)
        plt.axis('off')
        img_count += 1

        for lvl in range(1,6):
            filter_images = {}
            for blur_type in blur_mapping:
                kernels_for_level = blur_mapping[blur_type].get(lvl, [])
                if not kernels_for_level:
                    continue
                k = kernels_for_level[len(kernels_for_level)//2]
                temp = img_pre.copy()
                for x1, y1, x2, y2 in faces_pre:
                    face = temp[y1:y2, x1:x2]
                    if blur_type == 'gaussian':
                        face = cv2.GaussianBlur(face, (k,k), 0)
                    elif blur_type == 'average':
                        face = cv2.blur(face, (k,k))
                    elif blur_type == 'median':
                        face = cv2.medianBlur(face, k)
                    temp[y1:y2, x1:x2] = face
                filter_images[blur_type] = temp

            for blur_type, img in filter_images.items():
                plt.subplot(total_rows, total_cols, img_count)
                plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                plt.title(f'{blur_type.capitalize()} Level {lvl}')
                plt.axis('off')
                img_count += 1

    else:
        plt.figure(figsize=(18, 12))
        plt.subplot(3, 3, 1)
        plt.imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        plt.title('Original')
        plt.axis('off')

        plt.subplot(3, 3, 2)
        plt.imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
        plt.title('Preprocessed')
        plt.axis('off')

        plt.subplot(3, 3, 4)
        plt.imshow(cv2.cvtColor(img_det_orig, cv2.COLOR_BGR2RGB))
        plt.title('Detected Faces (Original)')
        plt.axis('off')

        plt.subplot(3, 3, 5)
        plt.imshow(cv2.cvtColor(img_det_pre, cv2.COLOR_BGR2RGB))
        plt.title('Detected Faces (Preprocessed)')
        plt.axis('off')

        filter_images = {}
        for blur_type in blur_mapping:
            kernels_for_level = blur_mapping[blur_type].get(user_level, [])
            if not kernels_for_level:
                continue
            k = kernels_for_level[len(kernels_for_level)//2]
            temp = img_pre.copy()
            for x1, y1, x2, y2 in faces_pre:
                face = temp[y1:y2, x1:x2]
                if blur_type == 'gaussian':
                    face = cv2.GaussianBlur(face, (k,k), 0)
                elif blur_type == 'average':
                    face = cv2.blur(face, (k,k))
                elif blur_type == 'median':
                    face = cv2.medianBlur(face, k)
                temp[y1:y2, x1:x2] = face
            filter_images[blur_type] = temp

        for i, (blur_type, img) in enumerate(filter_images.items()):
            plt.subplot(3, 3, 7+i)
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            plt.title(f'{blur_type.capitalize()} Level {user_level}')
            plt.axis('off')

    plt.tight_layout()
    plt.show()
